
# Kompresi Gambar Menggunakan SVD (Singular Value Decomposition)

**Singular Value Decomposition (SVD)** adalah salah satu teknik aljabar linier paling kuat yang digunakan dalam pemrosesan sinyal dan gambar. Salah satu aplikasi utamanya adalah **kompresi gambar** (image compression), yang bertujuan untuk mengurangi ukuran penyimpanan gambar dengan tetap mempertahankan kualitas visual yang dapat diterima.

---

## 1. Bagaimana Gambar Direpresentasikan dalam Matematika?

Sebelum memahami SVD, kita harus tahu bahwa dalam dunia digital, sebuah gambar grayscale (hitam-putih) tidak lebih dari sebuah **matriks berukuran $m \times n$**.
* Setiap elemen di dalam matriks tersebut disebut **piksel**.
* Nilai elemen matriks merepresentasikan intensitas cahaya (biasanya berkisar antara 0 untuk hitam hingga 255 untuk putih).
* Untuk gambar berwarna (RGB), gambar tersebut direpresentasikan oleh tiga buah matriks (lapisan Merah, Hijau, dan Biru).

---

## 2. Apa itu Singular Value Decomposition (SVD)?

Secara matematis, SVD menyatakan bahwa setiap matriks real $A$ berukuran $m \times n$ dapat difaktorkan menjadi perkalian tiga matriks:

$$A = U \Sigma V^T$$

Dimana:
1. **$U$ (Matriks Ortogonal Kiri):** Berukuran $m \times m$. Kolom-kolom di dalam $U$ disebut *left-singular vectors*, yang merepresentasikan fitur geometri vertikal dari gambar.
2. **$\Sigma$ (Matriks Diagonal):** Berukuran $m \times n$. Elemen diagonalnya ($\sigma_1, \sigma_2, \dots, \sigma_k$) disebut **singular values** (nilai singular). Nilai-nilai ini selalu non-negatif dan diurutkan dari yang terbesar ke yang terkecil ($\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_k \ge 0$). Nilai singular ini merepresentasikan "energi" atau tingkat kepentingan informasi dari gambar.
3. **$V^T$ (Transpose Matriks Ortogonal Kanan):** Berukuran $n \times n$. Kolom-kolom dari $V$ (*right-singular vectors*) merepresentasikan fitur geometri horizontal dari gambar.

---

## 3. Mekanisme Kompresi Gambar dengan SVD

Rahasia kompresi SVD terletak pada matriks diagonal $\Sigma$. Karena nilai singular diurutkan dari yang terbesar ke yang terkecil, nilai-nilai pertama menyimpan informasi struktural utama dari gambar, sedangkan nilai-nilai terakhir biasanya hanya menyimpan detail halus atau *noise* (derau).

Kita bisa melakukan kompresi dengan teknik bernama **Low-Rank Approximation** (Pendekatan Rank Rendah). Kita hanya mengambil $k$ nilai singular pertama yang terbesar, dan mengabaikan sisa nilai singular lainnya.

Formula untuk rekonstruksi gambar terkompresi ($A_k$) adalah:

$$A_k = \sum_{i=1}^{k} \sigma_i u_i v_i^T$$

Dimana $u_i$ adalah kolom ke-$i$ dari $U$, dan $v_i^T$ adalah baris ke-$i$ dari $V^T$.

### Perbandingan Penyimpanan Data:
* **Tanpa SVD (Gambar Asli):** Membutuhkan penyimpanan sebesar $m \times n$ angka.
* **Dengan SVD (Rank-$k$):** Kita hanya perlu menyimpan $k$ kolom pertama dari $U$, $k$ nilai singular dari $\Sigma$, dan $k$ baris pertama dari $V^T$. Total data yang disimpan adalah:
  $$\text{Total Data} = (m \times k) + k + (n \times k) = k(m + n + 1)$$

Jika kita memilih nilai $k$ yang jauh lebih kecil daripada $\min(m,n)$, maka ukuran data yang disimpan akan berkurang drastis!

---

## 4. Contoh Ilustrasi Kompresi

Bayangkan sebuah gambar berukuran $500 \times 500$ piksel (Total = 250.000 angka).

* **Jika $k = 10$ (Kompresi Sangat Tinggi):**
  * Ukuran data = $10 \times (500 + 500 + 1) = 10.010$ angka.
  * *Hasil:* Gambar terlihat sangat buram atau pixelated, namun bentuk dasarnya sudah mulai terlihat.
* **Jika $k = 50$ (Kompresi Sedang):**
  * Ukuran data = $50 \times (500 + 500 + 1) = 50.050$ angka (Hanya ~20% dari ukuran asli).
  * *Hasil:* Gambar sudah terlihat cukup jelas, detail utama sudah kembali, meskipun detail yang sangat kecil agak halus/hilang.
* **Jika $k = 100$ (Kualitas Bagus):**
  * Ukuran data = $100 \times (500 + 500 + 1) = 100.100$ angka (~40% dari ukuran asli).
  * *Hasil:* Secara visual hampir tidak bisa dibedakan dengan gambar asli oleh mata manusia.

---

## 5. Implementasi Code (Python dengan NumPy & OpenCV)

Berikut adalah contoh skrip Python sederhana untuk mencoba kompresi gambar menggunakan SVD:

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Load gambar dalam mode grayscale (hitam putih)
img = cv2.imread('nama_gambar_kamu.jpg', cv2.IMREAD_GRAYSCALE)

# 2. Terapkan SVD
U, S, Vt = np.linalg.svd(img, full_matrices=False)

# 3. Tentukan nilai k (jumlah komponen nilai singular yang dipertahankan)
k_values = [5, 20, 50, 100]

plt.figure(figsize=(12, 8))

# Tampilkan gambar asli
plt.subplot(2, 3, 1)
plt.title("Gambar Asli")
plt.imshow(img, cmap='gray')
plt.axis('off')

# Rekonstruksi gambar untuk setiap nilai k
for i, k in enumerate(k_values):
    # Ambil komponen k pertama
        U_k = U[:, :k]
    S_k = np.diag(S[:k])
    Vt_k = Vt[:k, :]
    
    # Rekonstruksi matriks gambar
    img_compressed = np.dot(U_k, np.dot(S_k, Vt_k))
    
    # Plot hasil
    plt.subplot(2, 3, i + 2)
    plt.title(f"Kompresi dengan k = {k}")
    plt.imshow(img_compressed, cmap='gray')
    plt.axis('off')

plt.tight_layout()
plt.show()